In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import math
import copy

# ==========================================
# 1. Data Preparation (CIFAR-10)
# ==========================================
def get_cifar10_loaders(batch_size=256):
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
    ])

    train_set = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
    test_set = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, num_workers=2)
    test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False, num_workers=2)

    return train_loader, test_loader

# ==========================================
# 2. Model Architectures & LoRA Components
# ==========================================
class MAE(nn.Module):
    """
    Masked Autoencoder Structure.
    """
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, padding=1, stride=2),
            nn.ReLU(),
            nn.Conv2d(64, 128, kernel_size=3, padding=1, stride=2),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten()
        )
        self.decoder = nn.Sequential(
            nn.Linear(128, 512),
            nn.ReLU(),
            nn.Linear(512, 3 * 32 * 32)
        )
        self.embedding_dim = 128

    def forward_encoder(self, x):
        return self.encoder(x)

    def forward(self, x):
        features = self.forward_encoder(x)
        reconstruction = self.decoder(features)
        return reconstruction.view(-1, 3, 32, 32)

class LoRA_Conv2d(nn.Module):
    """
    Custom LoRA wrapper for Conv2d layers.
    W' = W + B * A
    """
    def __init__(self, original_conv, r=4):
        super().__init__()
        self.original_conv = original_conv

        # Freeze the original convolution weights
        for param in self.original_conv.parameters():
            param.requires_grad = False

        # Matrix A (reduces dimension to rank 'r')
        self.lora_A = nn.Conv2d(
            original_conv.in_channels, r,
            kernel_size=original_conv.kernel_size,
            stride=original_conv.stride,
            padding=original_conv.padding,
            bias=False
        )

        # Matrix B (expands dimension back to out_channels)
        self.lora_B = nn.Conv2d(
            r, original_conv.out_channels,
            kernel_size=1, stride=1, padding=0, bias=False
        )

        # Initialization: A is Kaiming, B is zero (so initial LoRA impact is 0)
        nn.init.kaiming_uniform_(self.lora_A.weight, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B.weight)

    def forward(self, x):
        # original_output + LoRA_output
        return self.original_conv(x) + self.lora_B(self.lora_A(x))

class LoRAClassifier(nn.Module):
    """
    Injects LoRA layers into the MAE encoder and attaches a linear classification head.
    """
    def __init__(self, base_encoder, num_classes=10, r=4):
        super().__init__()
        # Deep copy to avoid mutating the pre-trained model directly
        self.encoder_module = copy.deepcopy(base_encoder)

        # Replace Conv2d layers with LoRA Conv2d layers
        # self.encoder_module.encoder[0] is the first Conv2d
        # self.encoder_module.encoder[2] is the second Conv2d
        self.encoder_module.encoder[0] = LoRA_Conv2d(self.encoder_module.encoder[0], r=r)
        self.encoder_module.encoder[2] = LoRA_Conv2d(self.encoder_module.encoder[2], r=r)

        # New trainable classification head
        self.fc = nn.Linear(base_encoder.embedding_dim, num_classes)

    def forward(self, x):
        features = self.encoder_module.forward_encoder(x)
        return self.fc(features)

# ==========================================
# 3. Pre-training Phase (Self-Supervised)
# ==========================================
def pretrain_mae(model, train_loader, device, epochs=5):
    print("\n--- Starting MAE Pre-training ---")
    model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.MSELoss()

    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for images, _ in train_loader:
            images = images.to(device)

            optimizer.zero_grad()
            reconstructed = model(images)

            loss = criterion(reconstructed, images)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)
        print(f"Pre-training Epoch [{epoch+1}/{epochs}] - MSE Loss: {avg_loss:.4f}")

# ==========================================
# 4. LoRA Evaluation Phase
# ==========================================
def lora_evaluation(encoder, train_loader, test_loader, device, epochs=10, r=4):
    print(f"\n--- Starting LoRA Evaluation (Rank r={r}) ---")
    probe_model = LoRAClassifier(encoder, num_classes=10, r=r).to(device)

    # Filter parameters to ONLY optimize ones that require gradients (LoRA A, LoRA B, and FC layer)
    trainable_params = filter(lambda p: p.requires_grad, probe_model.parameters())

    # Using Adam for LoRA fine-tuning generally converges better than SGD
    optimizer = optim.Adam(trainable_params, lr=1e-3, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()

    # Count trainable params vs total params to verify LoRA efficiency
    total_params = sum(p.numel() for p in probe_model.parameters())
    trained_params = sum(p.numel() for p in probe_model.parameters() if p.requires_grad)
    print(f"Trainable Parameters: {trained_params} / {total_params} ({(trained_params/total_params)*100:.2f}%)")

    # Train the LoRA modules and classification head
    probe_model.train()
    for epoch in range(epochs):
        total_loss = 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = probe_model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)
        print(f"LoRA Eval Epoch [{epoch+1}/{epochs}] - Classification Loss: {avg_loss:.4f}")

    # Evaluate accuracy
    print("\n--- Evaluating LoRA Accuracy ---")
    probe_model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = probe_model(images)

            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    print(f"Final LoRA Accuracy on CIFAR-10: {accuracy:.2f}%")
    return accuracy

# ==========================================
# Execution Block
# ==========================================
if __name__ == '__main__':
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")

    train_loader, test_loader = get_cifar10_loaders(batch_size=256)

    # 1. Initialize and Pre-train MAE
    mae_model = MAE()
    pretrain_mae(mae_model, train_loader, device, epochs=10)

    # 2. Evaluate using LoRA Adapter (Rank = 4)
    lora_evaluation(mae_model, train_loader, test_loader, device, epochs=15, r=4)

Using device: cuda


100%|██████████| 170M/170M [22:55<00:00, 124kB/s]



--- Starting MAE Pre-training ---
Pre-training Epoch [1/10] - MSE Loss: 0.9116
Pre-training Epoch [2/10] - MSE Loss: 0.7149
Pre-training Epoch [3/10] - MSE Loss: 0.6743
Pre-training Epoch [4/10] - MSE Loss: 0.6560
Pre-training Epoch [5/10] - MSE Loss: 0.6436
Pre-training Epoch [6/10] - MSE Loss: 0.6317
Pre-training Epoch [7/10] - MSE Loss: 0.6243
Pre-training Epoch [8/10] - MSE Loss: 0.6156
Pre-training Epoch [9/10] - MSE Loss: 0.6085
Pre-training Epoch [10/10] - MSE Loss: 0.6018

--- Starting LoRA Evaluation (Rank r=4) ---
Trainable Parameters: 1646454 / 1722102 (95.61%)
LoRA Eval Epoch [1/15] - Classification Loss: 2.0401
LoRA Eval Epoch [2/15] - Classification Loss: 1.8204
LoRA Eval Epoch [3/15] - Classification Loss: 1.7411
LoRA Eval Epoch [4/15] - Classification Loss: 1.7009
LoRA Eval Epoch [5/15] - Classification Loss: 1.6733
LoRA Eval Epoch [6/15] - Classification Loss: 1.6512
LoRA Eval Epoch [7/15] - Classification Loss: 1.6322
LoRA Eval Epoch [8/15] - Classification Loss: 1.6